# species_difference_map

Generates `species_difference_map.csv` for the Tableau dot map.

**Input:** `mosquito_suitability.csv` — ERA5 climate normals, one row per city × month  
**Output:** `species_difference_map.csv` — one row per city with season length difference

**Difference** = `season_albopictus − season_aegypti` (Moderate threshold, 0.3)  
- Positive → *Ae. albopictus* has longer season  
- Negative → *Ae. aegypti* has longer season  
- Zero → both species equally suitable

In [1]:
import pandas as pd

In [2]:
#  Config
INPUT_FILE  = "mosquito_suitability.csv"
OUTPUT_FILE = "species_difference_map.csv"
THRESHOLD   = 0.3  # Moderate threshold, consistent with analytical section

In [3]:
#  Load
df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df):,} rows — {df['city'].nunique():,} cities")

Loaded 17,076 rows — 1,418 cities


In [4]:
#  Season length per city per species
# Season length = number of months where suitability score >= threshold
df["suitable_aegypti"]    = (df["suitability_score_aegypti"]    >= THRESHOLD).astype(int)
df["suitable_albopictus"] = (df["suitability_score_albopictus"] >= THRESHOLD).astype(int)

city_cols = ["city", "country", "lat", "lon", "elevation_m"]
agg = df.groupby(city_cols, as_index=False).agg(
    season_aegypti    = ("suitable_aegypti",    "sum"),
    season_albopictus = ("suitable_albopictus", "sum")
)
agg.head()

,city,country,lat,lon,elevation_m,season_aegypti,season_albopictus
0,Aba,Nigeria,5.1167,7.3667,60.0,12,12
1,Abeokuta,Nigeria,7.1608,3.3483,85.0,12,12
2,Abhepur,India,39.4100,116.3800,25.0,5,4
3,Abidjan,Côte d’Ivoire,5.3364,-4.0267,41.0,12,12
4,Abomey-Calavi,Benin,6.4486,2.3556,13.0,12,12


In [5]:
#  Difference
agg["difference"] = agg["season_albopictus"] - agg["season_aegypti"]

In [6]:
#  Sanity checks
assert not agg.duplicated(["city", "country"]).any(), "Duplicate city+country detected"
assert agg["difference"].between(-12, 12).all(), "Difference out of expected range"

print(f"Difference range: {agg['difference'].min()} to {agg['difference'].max()}")
print(f"Albopictus longer: {(agg['difference'] > 0).sum():,} cities")
print(f"Aegypti longer:    {(agg['difference'] < 0).sum():,} cities")
print(f"Equal:             {(agg['difference'] == 0).sum():,} cities")

Difference range: -7 to 12
Albopictus longer: 348 cities
Aegypti longer:    601 cities
Equal:             474 cities


In [7]:
#  Save
agg.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}  ({len(agg)} rows, {list(agg.columns)})")

Saved: species_difference_map.csv  (1423 rows, ['city', 'country', 'lat', 'lon', 'elevation_m', 'season_aegypti', 'season_albopictus', 'difference'])
